In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import RFE, RFECV, VarianceThreshold
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
from collections import Counter
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS'] 
plt.rcParams['axes.unicode_minus'] = False  

In [ ]:
data = pd.read_excel('data/mor_cleaned.xlsx')  
print(data.head())
X1 = data.iloc[:, :-1]  
y = data.iloc[:, -1]   
feature_names = list(X1.columns)

In [ ]:
X = X1.to_numpy() 
print(X)

In [ ]:
def regression_feature_selection(X, y, 
                               variance_threshold=0.05,
                               corr_threshold=0.9,
                                ):
    #step 1
    results = {}
    original_features = list(range(X.shape[1]))

    selector_variance = VarianceThreshold(threshold=variance_threshold)
    X_filtered = selector_variance.fit_transform(X)
    features_remaining_1 = [i for i in range(X.shape[1]) 
                           if i in selector_variance.get_support(indices=True)]
    print(f"remain: {len(features_remaining_1)}")
    print(f"remain: {[feature_names[i] for i in features_remaining_1]}")
    print(features_remaining_1)

    merged_features1 = [feature_names[i] for i in features_remaining_1]
    merged_df1 = pd.concat([X1[merged_features1], y], axis=1)

    # step 2
    X_temp = X[:, features_remaining_1]  
    X_temp_df = pd.DataFrame(X_temp)
    corr_matrix = X_temp_df.corr().abs()
    upper_tri = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))#
    X_temp_df['target'] = y
    corr_matrix5 = X_temp_df.corr()
    target_XY = corr_matrix5['target'].drop('target')
    for col in upper_tri.columns:
        for idx in upper_tri.index:
            if pd.notna(upper_tri.loc[idx, col]) and upper_tri.loc[idx, col] > corr_threshold:
                if target_XY[idx] >= target_XY[col]:
                    high_corr_features.add(col)  
                else:
                    high_corr_features.add(idx)  
    
    features_remaining_2 = [features_remaining_1[i] for i in range(len(features_remaining_1)) 
                           if i not in high_corr_features]
    print(f"remain: {len(features_remaining_2)}")
    print(f"remain: {[feature_names[i] for i in features_remaining_2]}")

    merged_features2 = [feature_names[i] for i in features_remaining_2]
    merged_df2 = pd.concat([X1[merged_features2], y], axis=1)
    merged_df3 = X1[merged_features2]
    return merged_df3, merged_features2

In [ ]:
# 执行特征选择
X2, feature_names2 = regression_feature_selection(X, y, variance_threshold=0.05, corr_threshold=0.9)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X2, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()   
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import AdaBoostRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.neighbors import KNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.neural_network import MLPRegressor

models = {
    '3.Ridge': Ridge(alpha=1.0, random_state=42),
    '6.DecisionTree': DecisionTreeRegressor(max_depth=5, random_state=42),
    '8.RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
    '9.SVR': SVR(kernel='linear', C=1.0),
    '10.AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42),
    '11.GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    '12.XGBoost': XGBRegressor(n_estimators=100, random_state=42),
    '13.CatBoost': CatBoostRegressor(n_estimators=100, random_state=42, 
                                  learning_rate=0.1,
                                  early_stopping_rounds=50,
                                  verbose=False), 
    '14.LightGBM': LGBMRegressor(n_estimators=100, random_state=42, verbose=-1 ),
}

results = {}
feature_selection_results = {}
optimal_features_results = {}

In [ ]:
min_features = 5
max_features = 100
feature_range = range(min_features, max_features + 1)

for model_name, model in models.items():
    print(f"\n--- for : {model_name} ---")
    X_train_used = X_train_scaled
    X_test_used = X_test_scaled
    try:
        rfecv = RFECV(
            estimator=model,
            step=1,
            cv=3,  
            scoring='neg_mean_squared_error',  
            min_features_to_select=min_features,
            n_jobs=-1  
        )
        
        rfecv.fit(X_train_used, y_train)
        
        optimal_n_features = rfecv.n_features_
        print(f"  number: {optimal_n_features}")
        
        optimal_features_results[model_name] = {
            'optimal_n_features': optimal_n_features,
            'cv_scores': rfecv.cv_results_['mean_test_score'],
            'rfecv_model': rfecv
        }
        

        print(f"  num({optimal_n_features}) for RFE...")
        rfe = RFE(
            estimator=model, 
            n_features_to_select=optimal_n_features,
            step=1
        )
        
        rfe.fit(X_train_used, y_train)
        
        selected_features = rfe.support_
        feature_ranking = rfe.ranking_
        
        X_train_selected = rfe.transform(X_train_used)
        X_test_selected = rfe.transform(X_test_used)
        
        if model_name in [ 'Ridge',  'SVR']:
            final_model = type(model)(**model.get_params())
        else:
            final_model = type(model)(**model.get_params())
            
        final_model.fit(X_train_selected, y_train)
        y_pred = final_model.predict(X_test_selected)
        

        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        

        cv_scores = cross_val_score(final_model, X_train_selected, y_train, 
                                   cv=5, scoring='r2')
        mean_cv_score = np.mean(cv_scores)
        

        results[model_name] = {
            'MSE': mse,
            'R2': r2,
            'Mean_CV_Score': mean_cv_score,
            'Selected_Features': selected_features,
            'Feature_Ranking': feature_ranking,
            'Optimal_n_features': optimal_n_features,
            'RFE_Model': rfe,
            'Final_Model': final_model
        }
        
        feature_selection_results[model_name] = selected_features
        
        print(f"✓ finish - : {optimal_n_features}, MSE: {mse:.4f}, R²: {r2:.4f}, mean CV R²: {mean_cv_score:.4f}")
        
    except Exception as e:
        print(f"✗ wrong: {str(e)}")
        continue

In [ ]:

results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Optimal_n_features': [results[model]['Optimal_n_features'] for model in results],
    'MSE': [results[model]['MSE'] for model in results],
    'R2': [results[model]['R2'] for model in results],
    'Mean_CV_R2': [results[model]['Mean_CV_Score'] for model in results]
}).sort_values('R2', ascending=False)

print(results_df.round(4))

In [ ]:
results_df.to_csv('results_df.csv', index=False)

df_re = pd.DataFrame([results])
df_re.to_csv('results.csv', index=False)


In [ ]:
for model_name, model in models.items():
    print(f"\n--- for: {model_name} ---")
    
    want_model_result = results[model_name] 
    
    print(f"num: {want_model_result['Optimal_n_features']}")
    print(f"test R²: {want_model_result['R2']:.4f}")
    
    want_selected_features = want_model_result['Selected_Features']  
    want_selected_feature_names = [feature_names2[i] for i in range(len(feature_names2))   
                             if want_selected_features[i]]                           
    want_df = pd.concat([X2[want_selected_feature_names], y], axis=1)   
    
    filename = f"{model_name}best.xlsx"
    want_df.to_excel(filename, index=False) 
    print(f"ok: {filename}")